In [2]:
initial_prompt = """You are a scam detection classifier. Analyze video descriptions and classify them as scam-intended or normal.

## TASK
Determine whether a video description represents a scam-intended or normal (non-scam) video.
CRITICAL: Return exactly one valid JSON object and nothing else. No markdown, no extra text, no commentary.

## INPUT
You will receive a single text field named `video_description` (Korean or English) that may include:
- On-screen dialogues, captions/OCR text, banners, graphics, actions, or visual descriptions

IMPORTANT: Base your decision strictly on this text only. Do not infer, guess, or use external knowledge.

## OUTPUT FORMAT
You must return a JSON object with the following structure:

{
  "is_scam": true | false,
  "confidence": 0.0 ~ 1.0,
  "risk": "low" | "mid" | "high",
  "evidence": ["short verbatim phrase 1", "short verbatim phrase 2"],
  "explanation": "2-4 concise sentences summarizing rationale and risk."
}

## OBJECTIVE
Judge whether the content intends to induce viewers into fraudulent, deceptive, or illicit actions.

## SCAM PATTERNS (Fraud Indicators)
Classify as scam if one or more are present:
- Investment/financial lures
- Guaranteed or outsized/rapid profits
- Urgent "act now / limited time" prompts
- Requests for deposits, wallet transfers, fees, or "seed money"
- Requests for passwords, OTPs, account/ID numbers, or other personal data
- Phishing/login pages
- External funnels: Telegram/Kakao/WhatsApp/Line invites, QR codes, links, "reading rooms"
- Forged IDs/badges/licenses
- Authority/celebrity/institution impersonation
- Illegal gambling/trade operations
- Multi-level "recruitment" pitches
- Religious/psychological inducement for money/data/influence
- "Install this app/site to earn"; fake customer-service or withdrawal screens

## NORMAL PATTERNS (Non-Scam Defaults)
Treat as normal when clearly:
- Information/news/education or scam awareness without inducement
- Daily life/hobbies/entertainment, food/cooking
- Jobs/labor, product ads/branding, reviews without guarantees, payment/data requests, illegality, or external funnels

## DECISION RULES

1. Strong signals (any one ⇒ scam=true):
   - Guaranteed/outsized returns
   - Direct ask to pay/deposit/transfer
   - Direct ask for credentials/personal data
   - Explicit join/contact/funnel (Kakao/Telegram/QR/link/DM)
   - Authority/celebrity/institution impersonation
   - Illegal gambling/trade operations

2. Moderate signals (context needed):
   - Trading "picks", ROI talk, win rates, withdrawal balances, screen mockups, charts/targets, "must buy before [date]", aggressive hype, suspicious app/site install, reward/points promises
   - These may be normal if explicitly framed as warning/reporting/education and contain no inducement or data/payment request

3. Warnings/news/education:
   - Treat as normal unless there is simultaneous inducement, guaranteed profits, data/payment requests, illegal operation, or external funnel calls

4. Insufficient/ambiguous info:
   - Default to is_scam=false, confidence ≤ 0.35, risk="low"

5. Mixed scenes:
   - Judge net intent; if any part solicits money/data/external contact or promises guaranteed profits, classify as scam

## RISK LEVEL MAPPING

"high": 
- ≥2 strong signals, OR
- Any direct request for personal data/passwords/OTP/payment/transfer/deposit, OR
- Explicit external funnel contact

"mid": 
- Exactly 1 strong signal, OR
- Multiple coherent moderate signals pointing to inducement

"low": 
- Weak/ambiguous cues
- Educational/news/branding context plausible
- No asks or guarantees

## CONFIDENCE SCALE (0.0-1.0)

0.90-1.00: Multiple consistent strong signals
0.70-0.89: One strong or many aligned moderate signals
0.50-0.69: Mixed evidence; some scam cues
0.30-0.49: Faint/conflicting cues; likely normal
0.00-0.29: No usable scam cues

## EVIDENCE EXTRACTION
- Return 1-4 short, verbatim spans from `video_description` as "evidence" (original language)
- Do not paraphrase; no long passages, duplicates, or invented text

## SAFETY & VALIDATION
- Do not hallucinate brands, people, numbers, or claims not present in the input
- Judge strictly from the provided text
- Do not output markdown or commentary; output must be one JSON object only
- If unsure, default to normal with low risk and low confidence

## EXAMPLE

Input:
The video promises 300% guaranteed profit within two days and shows a QR code to join a Telegram group.

Output:
{
  "is_scam": true,
  "confidence": 0.95,
  "risk": "high",
  "evidence": ["300% guaranteed profit", "QR code to join a Telegram group"],
  "explanation": "The description contains explicit profit guarantees and directs users to an external Telegram contact, both clear scam indicators."
}
"""

In [3]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

# load omni model default, the default init_vision/init_audio/init_tts is True
# if load vision-only model, please set init_audio=False and init_tts=False
# if load audio-only model, please set init_vision=False
model = AutoModel.from_pretrained(
    'openbmb/MiniCPM-o-2_6',
    trust_remote_code=True,
    attn_implementation='sdpa', # sdpa or flash_attention_2
    torch_dtype=torch.bfloat16,
    init_vision=True,
    init_audio=True,
    init_tts=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.eval().cuda()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-o-2_6', trust_remote_code=True)

# In addition to vision-only mode, tts processor and vocos also needs to be initialized
model.init_tts()


/home1/sun5676/miniconda3/envs/abnormality/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home1/sun5676/miniconda3/envs/abnormality/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home1/sun5676/miniconda3/envs/abnormality/lib/python3.10/site-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Loading checkpoint

In [4]:
from transformers import AutoTokenizer
from collections import Counter
import re

# 1) 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("openbmb/MiniCPM-o-2_6", trust_remote_code=True)


# 2) 토큰화 정보 전체 확인 함수
def inspect_tokens(prompt: str):
    # 원본 토큰 분해
    tokens = tokenizer.tokenize(prompt)

    # 토큰 ID (모델이 보는 숫자 시퀀스)
    token_ids = tokenizer(prompt)["input_ids"]

    # 토큰 빈도 (어떤 토큰이 많이 등장하는지)
    freq = Counter(tokens)

    print("=== 1) 전체 토큰 리스트 (순서 그대로) ===")
    print(tokens)

    print("\n=== 2) 토큰 ID 시퀀스 ===")
    print(token_ids)

    print("\n=== 3) 토큰 빈도 (상위 20개) ===")
    for tok, count in freq.most_common(20):
        print(f"{tok} : {count}")

    print("\n=== 4) 토큰 수 ===")
    print(len(token_ids))


# 3) 의미 있는 토큰 추출
def extract_meaningful_tokens(prompt: str):
    tokens = tokenizer.tokenize(prompt)
    # 의미 없는 패딩/구두점/짧은 토큰 제거 
    meaningful = [
        t for t in tokens
        if len(t) > 2                # Ġ" 와 같이 띄어쓰기와 결합된 기호 제거 위해 3자리부터 카운트
        and not re.fullmatch(r"[.,!?;:\-+(){}\[\]]", t)   
        and not re.fullmatch(r"<.*?>", t)                 
    ]
    freq = Counter(meaningful)
    print("의미 있는 토큰 중 빈도 수 정렬")
    for tok, count in freq.most_common(20):
        print(f"{tok} : {count}")




inspect_tokens(initial_prompt)
extract_meaningful_tokens(initial_prompt)

=== 1) 전체 토큰 리스트 (순서 그대로) ===
['You', 'Ġare', 'Ġa', 'Ġscam', 'Ġdetection', 'Ġclassifier', '.', 'ĠAnaly', 'ze', 'Ġvideo', 'Ġdescriptions', 'Ġand', 'Ġclassify', 'Ġthem', 'Ġas', 'Ġscam', '-int', 'ended', 'Ġor', 'Ġnormal', '.ĊĊ', '##', 'ĠTASK', 'Ċ', 'D', 'etermine', 'Ġwhether', 'Ġa', 'Ġvideo', 'Ġdescription', 'Ġrepresents', 'Ġa', 'Ġscam', '-int', 'ended', 'Ġor', 'Ġnormal', 'Ġ(', 'non', '-s', 'cam', ')', 'Ġvideo', '.Ċ', 'CR', 'ITICAL', ':', 'ĠReturn', 'Ġexactly', 'Ġone', 'Ġvalid', 'ĠJSON', 'Ġobject', 'Ġand', 'Ġnothing', 'Ġelse', '.', 'ĠNo', 'Ġmarkdown', ',', 'Ġno', 'Ġextra', 'Ġtext', ',', 'Ġno', 'Ġcommentary', '.ĊĊ', '##', 'ĠINPUT', 'Ċ', 'You', 'Ġwill', 'Ġreceive', 'Ġa', 'Ġsingle', 'Ġtext', 'Ġfield', 'Ġnamed', 'Ġ`', 'video', '_description', '`', 'Ġ(', 'K', 'orean', 'Ġor', 'ĠEnglish', ')', 'Ġthat', 'Ġmay', 'Ġinclude', ':Ċ', '-', 'ĠOn', '-screen', 'Ġdialog', 'ues', ',', 'Ġcaptions', '/', 'OCR', 'Ġtext', ',', 'Ġbanners', ',', 'Ġgraphics', ',', 'Ġactions', ',', 'Ġor', 'Ġvisual', 'Ġdescriptions'